In [ ]:
import sys
import numpy as np

sys.path.append('../../../src/')
from Rain.Rain import Rain
sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

In [ ]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [ ]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
          "num_of_workers": 1,
          "subscription_id":'6e14c264-a7fc-4db4-a23a-d972c21a2d99', # Menna's ID
          "location": 'eastus',
          "ips": ['40.87.71.244'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
          "ports": [50151]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 16,
  }
}

In [ ]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [ ]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [ ]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [ ]:
model = create_model()
rain = Rain(config, model)

In [ ]:
# model = rain.train(X_train, y_train, strategy='async')

In [ ]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [ ]:
model = rain.train(X_train, y_train, strategy='sync')

In [ ]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))